# E4 · QLoRA rhetoric model

1. Cut TED transcripts into 120-220-word segments.
2. **Silver labels**: a larger instruct model fills the rubric JSON (`pipeline.content.RUBRIC_PROMPT`),
   validated by the same pydantic schema the app uses.
3. Export 100 segments to `hand_check.csv` → you correct them → `gold.jsonl` (the test set; never trained on).
4. QLoRA fine-tune **Llama-3.2-3B-Instruct** (4-bit NF4, LoRA r=16) on the silver labels.
5. Compare against the same base model with prompting only (eval/rhetoric_eval.py does the same locally).

T4 runtime: silver labelling ~1 h for 500 segments (7B teacher, 4-bit), fine-tune ~1 h.
Llama 3.2 requires accepting Meta's licence on Hugging Face and an `HF_TOKEN` secret.
**Data licence** – TED talks are CC BY-NC-ND; the Kaggle "TED Talks" transcript set (Banik 2017) is for research; cite both.

In [ ]:
REPO = "https://github.com/<your-user>/vaktaai.git"
from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vaktaai/e4"
!mkdir -p $OUT
!git clone -q $REPO /content/vaktaai || echo "repo already present"
%cd /content/vaktaai
!pip -q install transformers==4.46.3 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.1.1 datasets pydantic==2.10.4 pyyaml
import os, sys, json, random, re
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
sys.path[:0] = ["/content/vaktaai/backend", "/content/vaktaai"]
from pipeline.content import RUBRIC_PROMPT, RhetoricReport, extract_json, DEVICE_TYPES
SEED = 13; random.seed(SEED)
BASE = "meta-llama/Llama-3.2-3B-Instruct"           # or "mistralai/Mistral-7B-Instruct-v0.3"
TEACHER = "Qwen/Qwen2.5-7B-Instruct"                # silver-label teacher (any stronger instruct model)

## 1 · Segments
Upload `transcripts.csv` from the Kaggle dataset *rounakbanik/ted-talks* to `MyDrive/vaktaai/e4/`.

In [ ]:
import pandas as pd
df = pd.read_csv(f"{OUT}/transcripts.csv")
def segments(text, lo=120, hi=220):
    sents = re.split(r"(?<=[.?!])\s+", re.sub(r"\([^)]*\)", "", text))  # drop (Laughter) etc.
    cur, out = [], []
    for s in sents:
        cur.append(s)
        n = sum(len(x.split()) for x in cur)
        if n >= lo:
            if n <= hi: out.append(" ".join(cur))
            cur = []
    return out
pool = [seg for t in df["transcript"].dropna().sample(400, random_state=SEED) for seg in segments(t)[:2]]
random.shuffle(pool)
SEGMENTS = pool[:520]
print(len(SEGMENTS), "segments; example:\n", SEGMENTS[0][:300])

## 2 · Silver labels from the teacher (validated with the app's pydantic schema)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
tok_t = AutoTokenizer.from_pretrained(TEACHER)
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, quantization_config=bnb, device_map="auto")

def chat(model, tok, text, max_new_tokens=900):
    msgs = [{"role": "system", "content": RUBRIC_PROMPT}, {"role": "user", "content": text}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

silver_path = f"{OUT}/silver.jsonl"
done = {json.loads(l)["text"] for l in open(silver_path)} if os.path.exists(silver_path) else set()
with open(silver_path, "a") as fh:
    for i, seg in enumerate(SEGMENTS):
        if seg in done: continue
        for attempt in range(2):
            try:
                label = RhetoricReport.model_validate(extract_json(chat(teacher, tok_t, seg)))
                fh.write(json.dumps({"text": seg, "label": label.model_dump(exclude={"devices": {"__all__": {"start", "end"}}, "transitions": {"__all__": {"start"}}})}) + "\n")
                break
            except Exception as exc:
                print(i, "retry", type(exc).__name__)
        if i % 25 == 0: print(i)
del teacher; torch.cuda.empty_cache()

## 3 · Hand-check set (100) → gold
Download `hand_check.csv`, correct `label_json` per row (keep valid JSON), save as `hand_checked.csv`, upload, run the next cell.

In [ ]:
silver = [json.loads(l) for l in open(silver_path)]
random.Random(SEED).shuffle(silver)
gold_candidates, train_rows = silver[:100], silver[100:]
pd.DataFrame([{"text": r["text"], "label_json": json.dumps(r["label"])} for r in gold_candidates]).to_csv(f"{OUT}/hand_check.csv", index=False)
print(len(train_rows), "training rows; hand_check.csv written")

In [ ]:
hc = f"{OUT}/hand_checked.csv"
src = pd.read_csv(hc) if os.path.exists(hc) else pd.read_csv(f"{OUT}/hand_check.csv")
print("using", "hand-checked" if os.path.exists(hc) else "UNCHECKED silver", "gold labels")
with open(f"{OUT}/gold.jsonl", "w") as fh:
    for _, r in src.iterrows():
        fh.write(json.dumps({"text": r["text"], "label": RhetoricReport.model_validate(json.loads(r["label_json"])).model_dump()}) + "\n")

## 4 · QLoRA fine-tune

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
base = prepare_model_for_kbit_training(base)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(base, lora); model.print_trainable_parameters()

MAX_LEN = 2048
def encode(row):
    msgs = [{"role": "system", "content": RUBRIC_PROMPT}, {"role": "user", "content": row["text"]}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    answer = json.dumps(row["label"], ensure_ascii=False) + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]; a_ids = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]  # loss only on the JSON answer
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

ds = Dataset.from_list(train_rows).map(encode, remove_columns=["text", "label"])
def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda xs, v: [x + [v] * (n - len(x)) for x in xs]
    return {"input_ids": torch.tensor(pad([b["input_ids"] for b in batch], tok.pad_token_id)),
            "attention_mask": torch.tensor(pad([b["attention_mask"] for b in batch], 0)),
            "labels": torch.tensor(pad([b["labels"] for b in batch], -100))}
args = TrainingArguments(output_dir="/content/qlora", per_device_train_batch_size=2, gradient_accumulation_steps=8,
                         num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
                         logging_steps=10, save_strategy="no", fp16=True, seed=SEED, report_to=[])
Trainer(model=model, args=args, train_dataset=ds, data_collator=collate).train()
model.save_pretrained(f"{OUT}/adapter"); tok.save_pretrained(f"{OUT}/adapter")

## 5 · Evaluate on gold: fine-tuned vs prompting only

In [ ]:
gold = [json.loads(l) for l in open(f"{OUT}/gold.jsonl")]
def score(predict):
    valid, f1s, maes = 0, {t: [] for t in DEVICE_TYPES}, []
    for row in gold:
        try:
            pred = RhetoricReport.model_validate(extract_json(predict(row["text"]))); valid += 1
        except Exception:
            continue
        g = RhetoricReport.model_validate(row["label"])
        for t in DEVICE_TYPES:
            p_has, g_has = any(d.type == t for d in pred.devices), any(d.type == t for d in g.devices)
            if p_has or g_has: f1s[t].append(2 * (p_has and g_has) / (p_has + g_has))
        maes.append(sum(abs(getattr(pred.appeals, k) - getattr(g.appeals, k)) for k in ("ethos", "pathos", "logos")) / 3)
    return {"json_valid": valid / len(gold), "appeal_mae": sum(maes) / max(len(maes), 1),
            **{f"f1_{t}": sum(v) / len(v) for t, v in f1s.items() if v}}

model.eval()
tuned = score(lambda text: chat(model, tok, text))
with model.disable_adapter():
    prompting = score(lambda text: chat(model, tok, text))
res = pd.DataFrame({"qlora": tuned, "prompting_only": prompting}); print(res)
res.to_csv(f"{OUT}/e4_results.csv")

## 6 · Export for local serving
Merged fp16 weights → `models/rhetoric/merged` on your laptop, then:
`python training/serve_rhetoric.py --model models/rhetoric/merged --port 8001` and set `VAKTA_LLM_URL=http://127.0.0.1:8001`.
(Ollama alternative: convert the merged model to GGUF with llama.cpp and point `VAKTA_LLM_URL` at Ollama.)

In [ ]:
from peft import PeftModel
full = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(full, f"{OUT}/adapter").merge_and_unload()
merged.save_pretrained(f"{OUT}/merged", safe_serialization=True); tok.save_pretrained(f"{OUT}/merged")
print("download", f"{OUT}/merged", "-> models/rhetoric/merged ; e4_results.csv -> eval/results/")